# Cricket Batting Stroke Classifier Training
### Google Colab / Kaggle Notebook

This notebook trains an ML model for classifying cricket batting shots into classes:
- **cover_drive**
- **straight_drive**
- **pull_shot**
- **cut_shot**
- **forward_defense**
- **sweep_shot**
- **flick_shot**

Data sources:
- Hugging Face datasets: `vicky521/cricket-shot`, `rokmr/cricket-shot` (streamed remotely)
- CricShot10 dataset (GitHub repository index)
- Pose Landmark Temporal Classifier (ST-GCN / 1D-CNN-LSTM on 33 MediaPipe landmarks over time)

Exports to **ONNX** and **TF.js** for seamless integration into the Cricket Analysis Engine.

In [ ]:
# 1. Environment Setup
!nvidia-smi
!pip install -q torch torchvision torchaudio datasets mediapipe onnx onnxruntime scikit-learn

In [ ]:
# 2. Stream Data from Hugging Face & Dynamically Inspect Columns
from datasets import load_dataset, get_dataset_split_names

HF_DATASET_ID = "vicky521/cricket-shot"
splits = get_dataset_split_names(HF_DATASET_ID)
print("Available splits:", splits)

# Streaming mode to avoid downloading massive video archives
stream_data = load_dataset(HF_DATASET_ID, split=splits[0], streaming=True)
sample_iter = iter(stream_data)
first_elem = next(sample_iter)
print("Dynamic schema columns detected:", list(first_elem.keys()))

In [ ]:
# 3. Biomechanical Landmark Temporal Classifier Architecture
import torch
import torch.nn as nn

class PoseTemporalShotClassifier(nn.Module):
    """
    Temporal Convolutional + Bidirectional GRU network for cricket shot classification.
    Input: (Batch, SequenceLength=45 frames, Features=33 landmarks * 3 coords = 99)
    Output: (Batch, NumClasses=7)
    """
    def __init__(self, in_features: int = 99, hidden_dim: int = 128, num_classes: int = 7):
        super().__init__()
        self.spatial_conv = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.temporal_gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x shape: (B, T, F)
        B, T, F = x.shape
        x_flat = x.view(B * T, F)
        feat = self.spatial_conv(x_flat)
        feat_seq = feat.view(B, T, -1)
        gru_out, _ = self.temporal_gru(feat_seq)
        # Pooling over time sequence (attention or mean pooling)
        context = torch.mean(gru_out, dim=1)
        logits = self.classifier(context)
        return logits

model = PoseTemporalShotClassifier()
print(model)

In [ ]:
# 4. Training Loop with CrossEntropyLoss and AdamW
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"Training configured on {device}")

In [ ]:
# 5. Export Trained Model to ONNX
dummy_input = torch.randn(1, 45, 99).to(device)
onnx_path = "cricket_shot_classifier.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["pose_sequence"],
    output_names=["shot_probabilities"],
    dynamic_axes={"pose_sequence": {0: "batch_size", 1: "sequence_length"}, "shot_probabilities": {0: "batch_size"}},
    opset_version=14
)
print(f"Shot classifier exported successfully to: {onnx_path}")